# Проект предсказания вероятности отмены доставки заказа CargaPronto


Логистическая компания по доставке товаров CargaPronto выявила проблему в своем бизнесе, заключающуюся в том что 55% заказов доставляется с опозданием. Для решения данной проблемы необходимо разработать модель машинного обучения, предсказывающую вероятность отмены доставки заказа для внедрения в системе принятия решений операторами службы доставки.

## Содержание
1. <a href="#business_ml">Постановка задачи</a>
1. <a href="#libraries">Импорт библиотек, выполнение базовых настроек</a>
1. <a href="#data_load">Загрузка данных, первичное знакомство</a>
1. <a href="#eda">EDA</a>
1. <a href="#split">Разделение на выборки</a>
1. <a href="#baseline">Обучение базовой модели</a>
1. <a href="#geo_cluster">Кластеризация по признакам, связанным с местоположением</a>
1. <a href="#beh_cluster">Кластеризация по признакам профилей клиентов</a>
1. <a href="#enhanced_model">Обучение модели на новых признаках</a>
1. <a href="#extra">Дополнительное задание — подбор лучшего количества кластеров</a>
1. <a href="#test">Тестирование лучшей модели</a>
1. <a href="#conclusion">Выводы и рекомендации</a>


<a id="business_ml"></a>

## Постановка задачи

На языке машинного обучения решается задача бинарной классификации, при этом модель должна возвращать вероятность принадлежности к каждому классу:
- 0 — доставка выполнена в срок, задержки не было
- 1 — доставка задержана (late_delivery_risk = 1)

Задача относится к классу обучения с учителем, так как в исходных данных присутствует целевая переменная late_delivery_risk.

**Признаки и Feature Engineering**
Помимо стандартных числовых и категориальных признаков (режим доставки, категория товара, цена, временные характеристики заказа), в качестве ключевой гипотезы проекта проверяется следующее:

> Профиль клиента и его географическое положение несут дополнительную предсказательную силу, которую нельзя извлечь из характеристик отдельного заказа напрямую.

Для проверки этой гипотезы применяется двухуровневая кластеризация:

- Географическая сегментация — алгоритм KMeans++ разбивает клиентов на логистические зоны по координатам (customer_lat, customer_lon), формируя признак geo_cluster. Предположение состоит в том, что задержки распределены неравномерно по территории: одни зоны обслуживаются надёжнее, другие — системно хуже.

- Поведенческая сегментация (RFM) — клиенты группируются по пяти показателям активности и благонадёжности: давности последнего заказа (recency), частоте (total_orders = Frequency), суммарной выручке (total_sales = Monytary), доле проблемных заказов (return_rate) и среднему размеру скидки (avg_discount). Результат записывается в признак beh_cluster. Предположение состоит в том, что поведенческий профиль клиента коррелирует с риском задержки — например, клиенты с высоким return_rate чаще сталкиваются с логистическими проблемами.


**Модели**

В качестве основного алгоритма используется CatBoostClassifier — как хорошо зарекомендовавшая себя модель, встроенной поддержкой категориальных признаков и низкой склонностью к переобучению. В качестве дополнительного алгоритма используется LogisticRegression — как интерпретируемая линейная базовая модель, позволяющая оценить, насколько зависимости в данных поддаются линейному описанию.

Обе модели обучаются дважды: на базовом наборе признаков (только характеристики заказа) и на расширенном наборе (с добавлением geo_cluster и beh_cluster). Это позволяет количественно оценить вклад кластеризации в качество прогноза.

**Метрики оценки**

Модель оценивается по метрике ROC-AUC как основной, поскольку она устойчива к дисбалансу классов и отражает способность модели ранжировать заказы по вероятности задержки. Дополнительно анализируются стандартные метрики классификации: Precision, Recall и F1-score.


**Модель считается успешной при выполнении следующего условия:**

- ROC-AUC на тестовой выборке ≥ 0.75
- Дополнительным критерием служит положительный прирост ROC-AUC расширенной модели (с кластерами) по сравнению с базовой (без кластеров) — это будет означать, что гипотеза о влиянии профиля клиента и географии на риск задержки подтверждена.

<a id="libraries"></a>

##  Импорт библиотек, выполнение базовых настроек

In [147]:
! rm requirements.txt
! python --version

Python 3.12.9


In [148]:
from pathlib import Path

requirements_file = Path('requirements.txt')
requirements = [
    'scikit-learn==1.6.1',
    'seaborn==0.13.2',
    'optuna==4.8.0',
    'humanfriendly==10.0',
    'phik==0.12.5',
    'catboost==1.2.10',
]
if not requirements_file.exists():
    with open(requirements_file, 'w') as f:
        f.write('\n'.join(requirements))
        print(f'{requirements_file} created')
else:
    print(f'{requirements_file} exists')

print('Установка зависимостей...')
!pip install -r requirements.txt
print('Зависимости успешно установлены!')


requirements.txt created
Установка зависимостей...
Зависимости успешно установлены!


Импорт классов и методов

In [149]:
import os
import requests
import optuna
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from phik import phik_matrix


from catboost import CatBoostClassifier

from sklearn.model_selection import TimeSeriesSplit, KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.model_selection import (
    cross_validate,
    cross_val_score,
)
from sklearn.metrics import (
    confusion_matrix,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    make_scorer,
)

from humanfriendly import format_size


### Загрузка библиотеки классов SoloNeiroGood

Персональная библиотека для решения задач DS за авторством Smirnov Nikolai Georgievich (c) 2026-2027

In [150]:
TARGET_COL_NAME = 'booking_status'
RANDOM_STATE = 42
NAN_FLOAT = -999999.0
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', None) # выводить все колонки
pd.set_option('display.max_colwidth', 500) # выводить больше символов в ячейке


class DataBaseEnvConfig:
    """
    Класс для конфигурации подключения к БД через переменные окружения
    """
    def get_engine(self):
        load_dotenv()
        connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
            os.getenv('DB_USER'),
            os.getenv('DB_PASSWORD'),
            os.getenv('DB_HOST'),
            os.getenv('DB_PORT'),
            os.getenv('DB_NAME')
        )
        return create_engine(connection_string)


#### Заргузка ресурсов
class ResourceLoader:
    """
    Класс для безопасной загрузки ресурсов по http или с помощью подключения к БД.
    По http можно загружать как датасет в формате CSV, так и прозвольный файл.
    Если файл уже был скачан ранее и сохранен в локальной файловой системе, то
    загрузка из удаленного источника не производится.

    Для загрузки датасета: load_dataset(dataset_url, local_file, local_path='datasets'): dataframe
    Для загрузки файла: load_resource(resource_url, local_path, local_file): path
    """

    def __init__(self):
        pass

    def load_dataset_sql(self, table_name, local_path='datasets', query=None):
        """
        Загрузка датасета из БД по SQL запросу
        """
        local_dataset = f'{local_path}/{table_name}.csv'

        if os.path.exists(local_dataset):
            df = pd.read_csv(local_dataset)
            print(f'Датасет загружен из локального файла {local_dataset} без подключений к БД')
        else:
            if query is None:
                query = f'SELECT * FROM {table_name}'
            print(f'Загружаем датасет из БД по SQL запросу: {query[:30]}...')
            load_dotenv()
            df = pd.read_sql_query(query, con=DataBaseEnvConfig().get_engine())
            # сохраним датасет локально, чтобы не загружать из файла при рестарте ноутбука
            os.makedirs(local_path, exist_ok=True)
            df.to_csv(local_dataset, index=False)
            df = pd.read_csv(local_dataset)
            file_size = format_size(os.path.getsize(local_dataset))
            print(f'Датасет успешно загружен в {local_dataset},',
                  f'размер файла: {file_size}')
        return df

    def load_resource(self, resource_url, local_path, local_file):
        """
        Загрузка произвольного файла из сети и сохраение его локально.
        Если файл уже существует локально, то загрузка не происходит
        """
        local_resource_file = f'{local_path}/{local_file}'
        if os.path.exists(local_resource_file):
            print(f'Файл {local_resource_file} уже существует')
            return local_resource_file

        os.makedirs(local_path, exist_ok=True)
        print(f'Загружаем файл из {resource_url}')
        response = requests.get(resource_url)
        if response.status_code == 200:
            with open(local_resource_file, 'wb') as f:
                f.write(response.content)
            file_size = format_size(os.path.getsize(local_resource_file))
            print(f'Файл успешно загружен в {local_resource_file},',
                  f'размер файла: {file_size}')
            return local_resource_file
        else:
            raise NetworkError(f'Ошибка при загрузке файла: {response.status_code}')


    def load_dataset(
        self,
        dataset_url,
        local_file = None,
        local_path='datasets',
        sep=',',
        decimal='.'
    ):
        """
        Специальный. метод загрузки датасета по сети. Если датасет уже был скачан ранее и
        сохранен в локальной файловой системе, то загрузка не происходит.

        Parameters:
            dataset_url: str - url датасета
            local_file: str - имя файла, куда будет сохранен датасет (если None - будет вычислен из URL)
            local_path: str - путь к папке, куда будет сохранен датасет
            sep: str - разделитель в файле CSV
            decimal: str - символ, разделяющий целую часть от дробной в CSV

        Returns:
            df: pd.DataFrame - датасет
        """
        if local_file is None:
            local_file = dataset_url.split('/')[-1]
        local_dataset_file = f'{local_path}/{local_file}'
        remote_dataset_url = dataset_url
        def read_dataset_csv():
            return pd.read_csv(local_dataset_file, sep=sep, decimal=decimal)

        try:
            df = read_dataset_csv()
            print(f'Датасет успешно загружен из {local_dataset_file}')
        except FileNotFoundError:
            self.load_resource(remote_dataset_url, local_path, local_file)
            df = read_dataset_csv()

        print(f'Размер загруженного датасета: {df.shape[0]} строк, {df.shape[1]} столбцов', )
        return df

#### EDA
class CorrelationDisplayer:
    """
    Класс для отображения матрицы корреляций признаков в разных видах
    """
    def __init__(self, corr_matrix):
        self.corr_matrix = corr_matrix

    def get_corr_matrix(self):
        """
        Возвращает матрицу корреляций признаков, переданную в конструкторе
        """
        return self.corr_matrix

    def _subset_corr_matrix(self, subset):
        subset_cols = self.corr_matrix.columns if subset is None else subset
        return self.corr_matrix.loc[subset_cols, subset_cols]

    def draw_corr_matrix_full(
        self,
        digits=2,
        title='Матрица корреляций признаков',
        subtitle=None,
        figsize=(16, 10),
        subset=None,
    ):
        """
        Отображает полную матрицу корреляций признаков в виде heatmap

        Parameters:
            digits: int - количество знаков после запятой в значениях heatmap
            subset: [] - модмножество столбцов кор.матрицы для отображения
        """
        plt.subplots(figsize=figsize)
        matrix = self._subset_corr_matrix(subset)
        sns.heatmap(matrix.round(digits), annot=True, cmap='coolwarm', linewidths=0.5)
        plt.title(title + "\n" + subtitle if subtitle else title)
        plt.show()


    def draw_corr_matrix_with_target(
            self,
            target_col,
            title='Матрица корреляций с таргетом',
            subtitle=None,
            figsize=(16, 10),
    ):
        """
        Отображает матрицу корреляций признаков с таргетом в виде вертикального heatmap

        Parameters:
            target_col: str - название столбца таргета
            title: str - заголовок графика
            subtitle: str - подзаголовок графика
        """
        plt.subplots(figsize=figsize)
        data_heatmap = self.corr_matrix.loc[
            self.corr_matrix.index != target_col
        ][[target_col]].sort_values(by=target_col, ascending=False)
        sns.heatmap(data_heatmap, annot=True, cmap='coolwarm', linewidths=0.5)
        plt.title(title + "\n" + subtitle if subtitle else title)
        plt.show()

    def draw_pair_correlations(self, subset=None, figsize=(16, 10), corr_threshold=0.9):
        """
        Отображает попарные корреляции признаков в виде вертикального heatmap

        Parameters:
            subset: [] - модмножество столбцов кор.матрицы для отображения
            figsize: (int, int) - размер графика
            corr_threshold: float - порог корреляции, ниже которого корреляция не будет отображена
        """
        # преобразуем матрицу корреляции в датафрейм попарных корреляций
        # feature_1+feature_2 -> correlation
        matrix = self._subset_corr_matrix(subset)
        pair_correlations = matrix \
            .stack() \
            .reset_index() \
            .rename(columns={
                'level_0': 'feature1',
                'level_1': 'feature2',
                0: 'correlation'
            }) \
            .query('feature1 != feature2') \
            .sort_values(by='correlation', ascending=False) \

        def order_pair(row):
            if row['feature1'] > row['feature2']:
                return row['feature2'] + '/' + row['feature1']
            else:
                return row['feature1'] + '/' + row['feature2']

        pair_correlations['order_pair'] = pair_correlations.apply(order_pair, axis=1)
        pair_correlations = pair_correlations.drop(columns=['feature1', 'feature2'])
        pair_correlations = pair_correlations.drop_duplicates().reset_index(drop=True)
        pair_correlations = pair_correlations.query('correlation > @corr_threshold')
        pair_correlations = pair_correlations.sort_values(by='correlation')
        pair_correlations.plot(
            x='order_pair',
            y='correlation',
            xlabel='Значение корреляции',
            ylabel='Пара признаков',
            kind='barh',
            legend=False,
            figsize=figsize,
            grid=True,
        )
        plt.title('Попарные корреляции')
        plt.show()
        return pair_correlations.sort_values(by='correlation', ascending=False).reset_index(drop=True)

class EDAHelper:
    """
    Класс для выполнения Exploratory Data Analysis (EDA)
    """
    def __init__(self):
        pass

    def df_info(self, df, name = '', n_samples=3):
        """
        Выводит информацию о датасете:
        - количество строк и столбцов
        - размер датасета в памяти
        - первые, случайные и последние n_samples строк
        - информацию о типах данных столбцов
        - информацию о пропущенных значениях и проценте пропущенных значений в каждом столбце

        Parameters:
            df: pd.DataFrame - датафрейм
            name: str - произвольное название датасета
            n_samples: int - количество строк для вывода объектов датасета из начала, середины и конца
        """
        print('-'*50)
        print(f'Описание датасета {name}:')
        print(f'Датасет {name} содержит {df.shape[0]} строк и {df.shape[1]} столбцов.')
        print(f'Размер датасета {name} в памяти: {format_size(df.memory_usage().sum())}')
        print('-'*50)
        print(f'Данные датасета {name}:')
        print('-'*50)
        sample_df =  pd.concat([
            df.head(n_samples).assign(place='head'),
            df.sample(n_samples, random_state=RANDOM_STATE).assign(place='random'),
            df.tail(n_samples).assign(place='tail'),
        ]).sort_index()
        # move place column to first place
        last_col = sample_df.pop(sample_df.columns[-1])
        sample_df.insert(0, last_col.name, last_col)
        display(sample_df)
        print('-'*50)

        print(df.info())
        nan_counts = self.na_info(df)
        if (len(nan_counts) > 0):
            display(nan_counts)
        else:
            print(f'В датасете {name} нет пропущенных значений')

    def convert_to_datetime(self, df, column, format='%Y-%m-%d', print_time_range=True):
        """
        Преобразует столбец в датафрейме в дату и отображает временную ось данных в столбце
        Parameters:
            df: pd.DataFrame - датафрейм
            column: str - название столбца
            format: str - формат даты
            print_time_range: bool - выводить ли временной диапазон данных в столбце
        """
        df[column] = pd.to_datetime(df[column], format=format)
        if print_time_range:
            self.print_time_range(df, column)
        return df


    def print_time_range(self, df, datetime_column):
        """
        Выводит временную ось данных в столбце
        Parameters:
            df: pd.DataFrame - датафрейм
            datetime_column: str - название столбца
        """
        # определим временную ось таблицы:
        start_date = df[datetime_column].min()
        end_date = df[datetime_column].max()
        diff = int((end_date - start_date) / np.timedelta64(1, 'D'))
        diff_years = diff / 365.25

        print(f'Данные в "{datetime_column}" представлены за период {diff} дн. ({diff_years:.1f} л.): {start_date} - {end_date}')

    def box_hist(self, df, column, title=None, bins=20, hue=None, kde=True, stat='density'):
        """
        Выводит boxplot и histogramm для столбца датасета на одном графике
        Parameters:
            df: pd.DataFrame - датафрейм
            column: str - название столбца
            title: str - заголовок графика
            bins: int - количество бинов в гистограмме
            hue: str - название столбца для разбивки гистограммы по категориям
            kde: bool - рисовать ли kernel density estimation (плотность распределения)
            stat: str - тип значения для оси Y:
                - 'density' - плотность распределения
                - 'count' - количество объектов
                - 'frequency' - частота
                - 'percent' - доля в процентах от всего датасета
        """
        f, (ax_box, ax_hist) = plt.subplots(2, sharex=True, gridspec_kw={"height_ratios": (.15, .85)})
        display(pd.DataFrame(df[[column]].describe().T))
        sns.boxplot(df[column], orient='h', ax=ax_box)
        sns.histplot(data=df, x=column, ax=ax_hist, bins=bins, hue=hue, kde=kde, stat=stat)

        f.suptitle(f'Распределение признака {column}' if title is None else title)
        if stat == 'density':
            ylabel = 'Плотность распределения'
        elif stat == 'count':
            ylabel = 'Количество'
        elif stat == 'frequecy':
            ylabel = 'Частота'
        else:
            ylabel = 'Доля'

        ax_box.set(xlabel='')
        ax_hist.set(
            xlabel=f'Значения признака {column}',
            ylabel=ylabel
        )
        plt.show()

    def time_line(self, df, datetime_column, y_column, ylabel=None, plot_kwargs={}):
        """
        Выводит значение признака y_column на временной оси datetime_column
        Parameters:
            df: pd.DataFrame - датафрейм
            datetime_column: str - название столбца со временной осью
            y_column: str - название столбца для отображения по оси Y
            ylabel: str - подпись оси Y
        """
        if ylabel is None:
            ylabel = y_column

        params = {
            'kind': 'line',
            'x': datetime_column,
            'y': y_column,
            'xlabel': 'Дата',
            'ylabel': ylabel,
            'title': f'Завимость признака "{y_column}" от времени',
            'legend': False,
        } | plot_kwargs
        df.plot(
            **params
        )
        plt.show()

    def drop_duplicates(self, df, subset=None):
        """
        Удаляет дубликаты из датасета и иценивает их количество и процент от датасета
        Parameters:
            df: pd.DataFrame - датафрейм
            subset: list - список столбцов, по которым проверять дубликаты
        """
        ndups = df.duplicated(subset=subset).sum()
        print(f'Найдено {ndups} дубликатов по {"всем" if subset is None else subset} столбцам')
        if ndups > 0:
            df_orig = df.copy()
            df.drop_duplicates(subset=subset, inplace=True)
            diff = len(df_orig) - len(df)
            diff_pct = diff / len(df_orig) * 100
            print(f'Удалено {diff} строк ({diff_pct:.1f}%) из {len(df_orig)}')
        else:
            print('Дубликатов не обнаружено')

    def na_info(self, df, round_digits=1):
        '''
        Возвращает таблицу с количеством и процентом пропусков в столбцах датасета.
        '''
        count_na_name = 'Количество пропусков'
        res = pd.DataFrame({
            'Количество строк': len(df),
            count_na_name: df.isna().sum(),
            'Процент пропусков': round(df.isna().mean()*100, round_digits)
        }).sort_values(by=count_na_name, ascending=False)
        return res.query(f'`{count_na_name}` > 0').reset_index()


    # Уникальные значения всех категориальных признаков
    def get_unique_values(self, df, top_n=5, cat_columns=[]):
        """
        Возвращает таблицу с уникальными значениями всех категориальных признаков.
        Parameters:
            df: pd.DataFrame - датафрейм
            top_n: int - количество значений для вывода высококардинальных столбцов
            cat_columns: list - список категориальных признаков, если не передан,
            то берутся все категориальные признаки с типом object
        """
        print('Уникальные значения всех категориальных признаков:\n')
        data = []
        columns = cat_columns if cat_columns else df.select_dtypes(include=['object']).columns
        for col in columns:
            unique_vals = df[col].unique().tolist()
            unique_vals.sort()
            top_n_vals = ', '.join(unique_vals[:top_n])
            unique_val_str =  top_n_vals if len(unique_vals) <= top_n else f'{top_n_vals}, ...'
            num_of_unique_vals = df[col].nunique()
            data.append({
                'feature': col,
                'num_of_unique_vals': num_of_unique_vals,
                'unique_vals': unique_val_str,
            })
        return pd.DataFrame(data)

### Обучение модели
class ModelTrainHelper:
    """
    Класс с утилитарными функциями для обучения и оценки качества модели МО
    """
    def __init__(self):
        self.best_estimator_ = None
        pass

    def do_cross_validation(
            self,
            pipelines,
            X_train_val, y_train_val,
            scoring,
            metrics_df_list = [],
            return_train_score=False,
            cv=5,
    ):
        """
        Выполняет обучение модели с помощью кросс-валидации.
        Parameters:
            pipelines: dict - словарь с конфигурациями моделей {model_name: {'model': model, 'fit_params': fit_params}}
            X_train_val: pd.DataFrame - тренировочная выборка
            y_train_val: pd.Series - тренировочный таргет
            scoring: list - список метрик для оценки качества модели (первая метрика - главная)
            metrics_df_list: list - внешний параметр для сохранения результатов в сводную таблицу
        """
        cv_results_by_model = {}
        # Обучение моделей
        for name, p in pipelines.items():
            start_time = time.time()
            cv_results = cross_validate(
                estimator=p['model'],
                X=X_train_val,
                y=y_train_val,
                scoring=scoring,
                return_train_score=return_train_score,
                return_estimator=True,
                cv=cv,
                verbose=0,
                n_jobs=-1,
                params=p.get('fit_params'),
            )
            end_time = time.time()
            wall_time = end_time - start_time
            print(f'Обучение "{name}" заняло {wall_time:.2f} секунд')
            cv_results['wall_time_sec'] = wall_time
            cv_results_by_model[name] = cv_results

        def non_negative_metric_(metric):
            """
            Утилитарная функция для определения положительной или отрицательной метрики и ее имени
            по приставке neg_ в названии метрики
            """
            if metric.startswith('neg_'):
                return (True, metric[len('_neg'):])
            else:
                return (False, metric)

        is_neg_main_metric, main_metric = non_negative_metric_(scoring[0])

        # Сохранение результатов в сводную таблицу:
        def append_metrics_(result_metrics, model_name, cv_results, test_or_train='test', scoring=[]):
            """
            Утилитарная функция для усреднения результатов метрики, полученной на кросс-валидации
            и сохранения ее в сводную таблицу
            """
            metrics_dict = {}

            metrics_dict['model_name'] = model_name
            metrics_dict['wall_time_sec'] = cv_results.get('wall_time_sec', 0)
            metrics_dict = metrics_dict | {
                metric: np.mean(cv_results[f'{test_or_train}_{metric}']) for metric in scoring
            }
            # for metric in scoring:
            #     print(metric, cv_results[f'{test_or_train}_{metric}'])
            # add standard deviation
            # metrics_dict = metrics_dict | {
            #     f'{metric}_std': np.std(-cv_results[f'{test_or_train}_{metric}']) for metric in scoring
            # }

            keys = list(metrics_dict.keys())
            for metric in keys:
                # invert sign for neg metrics like neg_mean_squared_error
                # and rename metrics withoud neg
                if metric.startswith('neg_'):
                    metrics_dict[metric[len('neg_'):]] = metrics_dict[metric] * -1 if not metric.endswith('_std') else 1
                    metrics_dict.pop(metric)

            result_metrics.append(metrics_dict)
            return metrics_dict

        for model_name, model_cv_results in cv_results_by_model.items():
            append_metrics_(
                metrics_df_list,
                model_name,
                model_cv_results,
                scoring=scoring,
            )
            if return_train_score and 'dummy' not in model_name.lower():
                append_metrics_(
                    metrics_df_list,
                    f'{model_name} (train)',
                    model_cv_results,
                    test_or_train='train',
                    scoring=scoring,
                )

        metrics_df = pd.DataFrame(metrics_df_list) \
            .sort_values(by=main_metric, ascending=is_neg_main_metric)

        metrics_df.set_index('model_name', inplace=True)
        return metrics_df.sort_index(axis=1)

    def confusion_matrix_displayed(self, y_true, y_pred, true_desc='Уйдет', false_desc='Останется'):
        """
        Визуализирует матрицу ошибок для бинарной классификации.
        y_true - истинные значения
        y_pred - предсказанные значения
        """
        cm = confusion_matrix(y_true, y_pred)

        # Визуализируем матрицу
        plt.figure(figsize=(8, 6))

        # Отображаем матрицу как изображение
        im = plt.imshow(cm, interpolation='nearest', cmap='Blues')
        plt.colorbar(im)

        # Добавляем подписи осей
        plt.xlabel('Предсказанные классы', fontsize=12)
        plt.ylabel('Истинные классы', fontsize=12)
        plt.title('Матрица ошибок', fontsize=14)

        # Настраиваем метки на осях
        tick_marks = [0, 1]
        plt.xticks(tick_marks, [false_desc, true_desc])
        plt.yticks(tick_marks, [false_desc, true_desc])

        # Матрица имеет структуру:
        matrix_desc = [
            ['TN', 'FP'],
            ['FN', 'TP'],
        ]

        # Добавляем числовые значения в ячейки
        for i in range(2):
            for j in range(2):
                plt.text(j, i, f'{matrix_desc[i][j]} {cm[i, j]}',
                        ha='center', va='center',
                        color='white' if cm[i, j] > cm.max()/2 else 'black',
                        fontsize=14)

        plt.tight_layout()
        plt.show()


        tn = cm[0, 0]
        fp = cm[0, 1]
        fn = cm[1, 0]
        tp = cm[1, 1]

        print('Расшифровка матрицы ошибок:')
        print(f'True Negatives (TN):  {tn} - правильно предсказали {false_desc}')
        print(f'False Positives (FP): {fp} - ошибочно предсказали {true_desc}')
        print(f'False Negatives (FN): {fn} - ошибочно предсказали {false_desc}')
        print(f'True Positives (TP):  {tp} - правильно предсказали {true_desc}')

    def feature_importance(self, model, feature_names):
        """
        Выводит важность признаков для линейной регрессии, строит bar plot важности.
        model - модель, для которой нужно вывести важность признаков
        feature_names - список названий признаков
        """
        # Получаем коэффициенты
        coefficients = model.coef_[0]
        intercept = model.intercept_[0]

        # DataFrame для анализа для удобства анализа коэффициентов
        coef_df = pd.DataFrame({
            'feature': feature_names,
            'coefficient': coefficients,
            'abs_coefficient': np.abs(coefficients)
        }).sort_values('abs_coefficient', ascending=False)

        # Визуализируем важность признаков:
        plt.figure(figsize=(8, 10))
        top_features = coef_df.sort_values(by='abs_coefficient', ascending=True)
        plt.barh(range(len(top_features)), top_features['coefficient'])
        plt.yticks(range(len(top_features)), top_features['feature'])
        plt.xlabel('Значение коэффициента')
        plt.title('Топ признаков по силе влияния на предсказание')
        plt.tight_layout()
        plt.show()

        return {
            'weights': coef_df.reset_index(drop=True),
            'intercept': intercept
        }

    def compare_metrics(self, baseline, enhanced, name, digits=3, pct_digits=0):
        """
        Сравнивает две метрики и выводит их разницу в процентах.
        baseline - базовая метрика
        enhanced - улучшенная метрика
        name - название метрики
        digits - количество знаков после запятой в числах
        pct_digits - количество знаков после запятой в процентах разницы
        """
        diff = enhanced - baseline
        diff_pct = diff / baseline * 100
        plus = '+' if diff > 0 else ''
        print(f'Улучшение метрики {name}:',
            f'{baseline:.{digits}f}->{enhanced:.{digits}f} ({plus}{diff_pct:.{pct_digits}f}%)')

class OptunaHelper:
    """
    Класс для оптимизации гиперпараметров с помощью Optuna.
    """
    def __init__(self, X, y, X_val=None, y_val=None, model_name='model'):
        self.X = X
        self.y = y
        self.X_val = X_val
        self.y_val = y_val
        self.model_name = model_name
        optuna.logging.set_verbosity(optuna.logging.WARNING)  # уменьшаем болтливость

    def fit_study(self, params_func, estimator_func, scorer_func=None, n_trials=30,
                  visualize=False, show_progress_bar=False, fit_score_func=None, cv=KFold(n_splits=5),
                  fit_params={}, fit_func=None, metrics_funcs={}):
        """
        Функция поиска оптимальных гиперпараметров.

        params_func - функция, возвращающая гиперпараметры для модели в формате Optuna
        estimator_func - функция, возвращающая модель с подготовленными гиперпараметрами
        scorer_func - функция, возвращающая метрику для оценки модели
        n_trials - число итераций
        visualize - флаг для визуализации результатов оптимизации
        show_progress_bar - флаг для вывода прогресса оптимизации
        fit_score_func - функция, осуществляющая вызов тренировки модели
        fit_func - функция, осуществляющая вызов тренировки модели
        metrics_funcs - словарь с функциями для вычисления метрик
        cv - объект кросс-валидации
        """

        def objective(trial):
            # описываем, какие гиперпараметры будем подбирать и в каких диапазонах.
            params = params_func(trial)

            # пайплайн с подготовкой данных и моделью
            # с подобранными на этой итерации гиперпараметрами
            pipeline = estimator_func(params)

            mean_score = 0
            if fit_score_func is not None:
                print('fit_score func was provided')
                mean_score = fit_score_func(pipeline, self.X, self.y, self.X_val, self.y_val, fit_params)
            elif isinstance(cv, KFold) or isinstance(cv, StratifiedKFold) or isinstance(cv, int):
                print('Using cross_val_score to optimize model')
                main_scores = cross_val_score(
                    pipeline,
                    self.X,
                    self.y,
                    cv=cv,
                    scoring=scorer_func,
                    n_jobs=-1
                )
            else:

                X = self.X
                y = self.y

                main_scores = []
                for fold, (train_index, valid_index) in enumerate(cv.split(X)):
                    # Разделение данных для текущего фолда
                    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index],
                    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

                    start_time = time.time()
                    if fit_func is not None:
                        X_valid_t = fit_func(pipeline, X_train, y_train, X_valid, y_valid, fit_params)
                        fit_time = time.time() - start_time
                        # Прогнозирование на уже трансформированных данных,
                        # чтобы не перезапускать pipeline (tfidf мог поменять словарь)
                        model = pipeline.named_steps['model']
                        score = scorer_func(model, X_valid_t, y_valid)
                    else:
                        pipeline.fit(X_train, y_train,
                                    eval_set=[(X_train, y_train), (X_valid, y_valid)],
                                    **fit_params)
                        fit_time = time.time() - start_time
                        score = scorer_func(pipeline, X_valid, y_valid)
                    main_scores.append(score)

                # Среднее значение метрики на кросс-валидации
                mean_score = np.mean(main_scores)

            # Сообщаем результат Optuna
            trial.report(mean_score, step=0)

            # Если результат плохой — прерываем итерацию
            if trial.should_prune():
                raise optuna.TrialPruned()

            # Возвращаем среднее значение метрики на кросс-валидации
            return mean_score

        # Фиксируем сид через семплер
        sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)

        study = optuna.create_study(
            direction='maximize',
            sampler=sampler,
        )

        study.optimize(
            objective,
            n_trials=n_trials, # число итераций
            show_progress_bar=show_progress_bar,
        )

        # Прогресс метрики по попыткам
        if (visualize):
            fig1 = optuna.visualization.plot_optimization_history(study)
            # Важность гиперпараметров
            fig2 = optuna.visualization.plot_param_importances(study)

            display(fig1)
            display(fig2)
        return study

class WallTime:
    """
    Класс для измерения времени выполнения кода.
    """
    def __init__(self):
        self.start_time = time.time()

    def elapsed_seconds(self):
        """
        Возвращает время в секундах, прошедшее с момента создания объекта.
        """
        return time.time() - self.start_time

class GridSearchHelper:
    """
    Класс для поиска оптимальных гиперпараметров с помощью GridSearchCV.
    """
    def __init__(self, X, y, scoring, cv):
        self.X = X
        self.y = y
        self.scoring = scoring
        self.main_metric = scoring[0]
        self.cv = cv
        self.last_results_top_ = None

    def fit_grid_(self, estimator, param_grid, model_name):
        print(f'Обучение модели {model_name} c перебором параметров...')
        grid = GridSearchCV(
            estimator=estimator,
            param_grid=param_grid,
            scoring=self.scoring,
            cv=self.cv,
            refit=self.main_metric,
            n_jobs=-1,
            verbose=0,
        )
        grid.fit(self.X, self.y)
        return grid

    def display_top_combinations_(self, grid_result, top_n=10):

        results_df = pd.DataFrame(grid_result.cv_results_) \
            .sort_values(by=f'mean_test_{self.main_metric}', ascending=False) \
            .reset_index(drop=True)

        print(f"\nТоп комбинаций по {self.main_metric}:")
        displayable_columns = [x for x in results_df.columns if x.startswith('mean_test_')]
        displayable_columns.insert(0, 'params')
        displayable_columns.append(f'std_test_{self.main_metric}')
        self.last_results_top_ = results_df[displayable_columns]
        display(self.last_results_top_.head(top_n))

    def fit_and_display_top(self, estimator, param_grid, model_name, top_n=5):
        """
        Подбор гиперпараметров и вывод топ комбинаций параметров и результатов

        estimator - модель
        param_grid - словарь с параметрами модели
        model_name - название модели
        top_n - количество лучших комбинаций параметров для вывода в итоговом датафрейме
        """
        grid_result = self.fit_grid_(estimator, param_grid, model_name)
        self.display_top_combinations_(grid_result, top_n=top_n)

        return grid_result

    def get_last_results_top(self):
        """
        Возвращает последние топ результатов поиска оптимальных гиперпараметров
        """
        return self.last_results_top_

<a id="data_load"></a>

## Загрузка данных, первичное знакомство

* Загрузите датасеты `ds_s18_customers.csv` и `ds_s18_orders.csv`.
  * Путь к первому файлу — `'/datasets/ds_s18_customers.csv'`.
  * Путь ко второму файлу — `'/datasets/ds_s18_orders.csv'`.
* Проведите технический аудит данных.
* Выявите и устраните полные дубликаты строк.
* Выполните проверку целостности данных (ID-Check).
* Подготовьте признаки, связанные с датой и временем.
* Создайте дополнительные колонки `order_month`, `order_weekday`, `order_hour`.




Загрузим датасеты для последующего локального чтения.

In [151]:
resource_loader = ResourceLoader()
df_customers = resource_loader.load_dataset('https://code.s3.yandex.net/datasets/ds_s18_customers.csv')
df_orders = resource_loader.load_dataset('https://code.s3.yandex.net/datasets/ds_s18_orders.csv')

Датасет успешно загружен из datasets/ds_s18_customers.csv
Размер загруженного датасета: 20652 строк, 8 столбцов
Датасет успешно загружен из datasets/ds_s18_orders.csv
Размер загруженного датасета: 180519 строк, 7 столбцов


Загружено 2 таблицы. в таблице пользователи 20к строк и 8 столбцов. В таблице заказы - 180к строк и 7 столбцов.

Проверим загруженные данные экспресс-анализом:

In [152]:
eda_helper = EDAHelper()
eda_helper.df_info(df_customers, 'customers')

--------------------------------------------------
Описание датасета customers:
Датасет customers содержит 20652 строк и 8 столбцов.
Размер датасета customers в памяти: 1.32 MB
--------------------------------------------------
Данные датасета customers:
--------------------------------------------------


,place,customer_id,customer_lat,customer_lon,total_sales,total_orders,avg_discount,return_rate,recency
0,head,1,25.953648,-97.507683,472.450012,1,0.060000,0.000000,793
1,head,2,38.375595,-104.726021,1618.660042,4,0.126000,0.000000,137
2,head,3,18.025375,-66.615082,3189.200037,5,0.105000,0.000000,230
341,random,345,36.958881,-122.032600,2642.310051,6,0.124615,0.153846,394
2408,random,2434,35.062752,-79.005745,1371.110027,4,0.123750,0.000000,217
8213,random,8280,18.242855,-66.370567,2016.150028,6,0.100769,0.230769,194
20649,tail,20755,18.251453,-66.037056,314.640015,1,0.040000,0.000000,1
20650,tail,20756,41.830791,-87.802979,10.910000,1,0.060000,0.000000,1
20651,tail,20757,25.860285,-80.197342,34.980000,1,0.120000,0.000000,1


--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 20652 entries, 0 to 20651
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   customer_id   20652 non-null  int64  
 1   customer_lat  20652 non-null  float64
 2   customer_lon  20652 non-null  float64
 3   total_sales   20652 non-null  float64
 4   total_orders  20652 non-null  int64  
 5   avg_discount  20652 non-null  float64
 6   return_rate   20652 non-null  float64
 7   recency       20652 non-null  int64  
dtypes: float64(5), int64(3)
memory usage: 1.3 MB
None
В датасете customers нет пропущенных значений


Таблица с пользователями не содержит пропусков. Типы столбцов не требуют модификации.

In [153]:
eda_helper.df_info(df_orders, 'orders')

--------------------------------------------------
Описание датасета orders:
Датасет orders содержит 180519 строк и 7 столбцов.
Размер датасета orders в памяти: 10.11 MB
--------------------------------------------------
Данные датасета orders:
--------------------------------------------------


,place,order_id,customer_id,late_delivery_risk,shipping_mode,category_name,order_date,item_price
0,head,180517,20755,0,Standard Class,Sporting Goods,2018-01-31 22:56:00,327.750000
1,head,179254,19492,1,Standard Class,Sporting Goods,2018-01-13 12:27:00,327.750000
2,head,179253,19491,0,Standard Class,Sporting Goods,2018-01-13 12:06:00,327.750000
19670,random,152658,12041,1,First Class,Women's Apparel,2017-06-09 18:43:00,50.000000
80120,random,78239,6109,1,Standard Class,Water Sports,2016-04-01 21:05:00,199.990005
114887,random,253,4271,0,Standard Class,Indoor/Outdoor Games,2015-01-02 14:32:00,49.980000
180516,tail,65129,291,1,Standard Class,Fishing,2016-01-15 21:00:00,399.980011
180517,tail,65126,2813,0,Standard Class,Fishing,2016-01-15 20:18:00,399.980011
180518,tail,65113,7547,0,Standard Class,Fishing,2016-01-15 18:54:00,399.980011


--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   order_id            180519 non-null  int64  
 1   customer_id         180519 non-null  int64  
 2   late_delivery_risk  180519 non-null  int64  
 3   shipping_mode       180519 non-null  str    
 4   category_name       180519 non-null  str    
 5   order_date          180519 non-null  str    
 6   item_price          180519 non-null  float64
dtypes: float64(1), int64(3), str(3)
memory usage: 9.6 MB
None
В датасете orders нет пропущенных значений


В таблице заказов так же нет пропусков. Данные загружены успешно.

Удалим явные дубликаты в данных при их наличии:

In [154]:
eda_helper.drop_duplicates(df_customers)

Найдено 0 дубликатов по всем столбцам
Дубликатов не обнаружено


In [155]:
eda_helper.drop_duplicates(df_customers, subset=['customer_id'])

Найдено 0 дубликатов по ['customer_id'] столбцам
Дубликатов не обнаружено


In [156]:
eda_helper.drop_duplicates(df_customers, ['customer_lat', 'customer_lon', 'total_sales', 'total_orders', 'avg_discount', 'return_rate', 'recency'])

Найдено 5 дубликатов по ['customer_lat', 'customer_lon', 'total_sales', 'total_orders', 'avg_discount', 'return_rate', 'recency'] столбцам
Удалено 5 строк (0.0%) из 20652


По подмножеству всех признаков кроме customer_id было удалено 5 строк из 20652 в таблице профилей пользователей.

In [157]:
eda_helper.drop_duplicates(df_orders)

Найдено 0 дубликатов по всем столбцам
Дубликатов не обнаружено


In [158]:
eda_helper.drop_duplicates(df_orders, ['order_id'])

Найдено 0 дубликатов по ['order_id'] столбцам
Дубликатов не обнаружено


In [159]:
eda_helper.drop_duplicates(df_orders, ['customer_id', 'late_delivery_risk', 'shipping_mode', 'category_name', 'order_date', 'item_price'])

Найдено 20806 дубликатов по ['customer_id', 'late_delivery_risk', 'shipping_mode', 'category_name', 'order_date', 'item_price'] столбцам
Удалено 20806 строк (11.5%) из 180519


По всем признакам кроме order_id из таблицы заказов удалено 20к (11.5%) строк с дублирующими данными, у которых совпадала даже дата и время заказа вплоть до секунд, что говорит о ошибки ввода или чтения данных из источника.

Выполним проверку целостности данных (ID-Check).

In [160]:
if set(df_customers['customer_id'].to_list()) == set(df_orders['customer_id'].to_list()):
    print('Все customer_id из df_customers присутствуют в df_orders')
else:
    print('Значения customer_id из df_customers, не совпадают с customer_id из df_orders!')

Значения customer_id из df_customers, не совпадают с customer_id из df_orders!


Выведем количество объектов из таблицы customers, которые отсутствуют в таблице orders:

In [161]:
(~df_customers['customer_id'].isin(df_orders['customer_id'])).sum()

np.int64(0)

Выведем количество заказов из таблицы orders, которые customer_id которых отсутствует в таблице customers:

In [162]:
not_in_customers = ~df_orders['customer_id'].isin(df_customers['customer_id'])
(not_in_customers).sum()

np.int64(5)

Таких записей 5 шт. Удалим их.

In [163]:
df_orders = df_orders[~not_in_customers]

In [164]:
if set(df_customers['customer_id'].to_list()) == set(df_orders['customer_id'].to_list()):
    print('Все customer_id из df_customers присутствуют в df_orders')
else:
    print('Значения customer_id из df_customers, не совпадают с customer_id из df_orders!')

Все customer_id из df_customers присутствуют в df_orders


Теперь идентификаторы заказчиков в обеих таблицах соответствуют.

Подготовим признаки, связанные с датой и временем. Создадим дополнительные колонки `order_month`, `order_weekday`, `order_hour`.

In [165]:
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'], format='%Y-%m-%d %H:%M:%S')

Напишем фунцию, осуществляющую фича инжениринг по столбцу с датой:

In [166]:
def date_features(df):
    df['order_month'] = df['order_date'].dt.month
    df['order_weekday'] = df['order_date'].dt.weekday
    df['order_hour'] = df['order_date'].dt.hour
    return df

df_orders = date_features(df_orders)
df_orders.sample(10, random_state=RANDOM_STATE)
display(df_orders.sample(10, random_state=RANDOM_STATE))
df_orders.drop('order_date', axis=1, inplace=True)


,order_id,customer_id,late_delivery_risk,shipping_mode,category_name,order_date,item_price,order_month,order_weekday,order_hour
114181,6505,2076,1,Second Class,Indoor/Outdoor Games,2015-02-07 20:47:00,49.980000,2,5,20
109438,109553,1708,1,Standard Class,Cleats,2016-10-02 07:30:00,59.990002,10,6,7
67630,91976,3553,1,Standard Class,Cleats,2016-06-22 00:59:00,59.990002,6,2,0
59701,42144,10199,0,Standard Class,Fishing,2015-09-04 04:33:00,399.980011,9,4,4
151134,99932,9138,0,Standard Class,Fishing,2016-08-07 13:53:00,399.980011,8,6,13
10523,151536,5727,0,Standard Class,Hunting & Shooting,2017-06-02 18:12:00,99.000000,6,4,18
87019,107036,8126,0,Standard Class,Shop By Sport,2016-09-17 16:28:00,39.990002,9,5,16
158565,1432,4695,1,Second Class,Cleats,2015-01-09 11:54:00,59.990002,1,4,11
62106,19020,2970,1,First Class,Water Sports,2015-04-21 20:30:00,199.990005,4,1,20
171470,117798,4933,0,Standard Class,Cardio Equipment,2016-11-18 22:41:00,99.989998,11,4,22


Фичи успешно добавлены.

<a id="eda"></a>

## EDA

На этом этапе ваша цель — исследовать структуру данных и подтвердить гипотезы транспортного департамента о «неоднородности» базы. Кроме того, вам предстоит изучить и удалить географические аномалии.

1. Проверьте баланс классов.
2. Выполните географический аудит.
3. Удалите аномалии.
4. Проанализируйте зависимости между признаками.



<a id="split"></a>

##  Разделение на выборки

Разделите датафрейм с заказами на выборки. В этом проекте вам предстоит  использовать «классическое» разделение на три выборки: обучающую, валидационную и тестовую. Использовать для этого нужно не `train_test_split`, а `GroupShuffleSplit`. Он делит данные так, чтобы группы, то есть клиенты, не пересекались.

Ниже приведён пример того, как отделить часть данных для обучения, используя `customer_id` как ключ для группировки.

```python
from sklearn.model_selection import GroupShuffleSplit

# 1. Выбираем колонку, по которой будем группировать (наши "группы")
groups = df_orders['customer_id']

# 2. Инициализируем сплиттер (например, отделим 20% клиентов для теста)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)

# 3. Получаем индексы для разделения
# Метод split возвращает генератор, поэтому используем next()
train_idx, test_idx = next(gss.split(df_orders, groups=groups))

# 4. Формируем выборки
df_train = df_orders.iloc[train_idx]
df_test = df_orders.iloc[test_idx]

```


1. Разделите данные из датафрейма с заказами  на три выборки  в соотношении 60:20:20.


2. После разделения обязательно убедитесь, что множества ID клиентов в `train`, `val` и `test` не пересекаются. Если один и тот же клиент попал в разные выборки, то разделение выполнено неверно.

<a id="baseline"></a>

## Обучение базовой модели

Прежде чем проверять гипотезу о сегментации и создавать при помощи кластеризации новые признаки, необходимо зафиксировать точку отсчёта. Вам нужно понять, какое качество прогноза задержек обеспечивают стандартные модели, используя только базовую информацию о самом заказе.

1. Выберите нужные признаки — используйте колонки таблицы `orders` (`shipping_mode`, `category_name`, `item_price`, `order_hour`,  `order_weekday` и `order_month`).

2. Обучите логистическую регрессию и `CatBoostClassifier` и оцените их качество.

3. Оцените качество обеих моделей на валидационной выборке.

Напоминаем, что на этапе построения базовой модели глубокий подбор гиперпараметров необязателен.

**Дополнительное задание.** Если вы чувствуете в себе силы, вы можете попробовать базовую оптимизацию, но помните: главная прибавка к качеству ожидается от новых признаков, которые вы создадите в следующих разделах.






<a id="geo_cluster"></a>

##  Кластеризация по признакам, связанным с местоположением



На этом этапе нужно превратить исходные координаты `customer_lat` и `customer_lon` в полезный для модели признак — логистическую зону.

**Ваши действия:**

1. Проведите корректную подготовку признаков.
2. Продумайте защиту от утечки данных.
3. Используйте метод локтя, чтобы найти математически обоснованную границу количества групп.
4. Постройте график с полученными кластерами и отметьте центроиды.

<a id="beh_cluster"></a>

##  Кластеризация по признакам профилей клиентов





1. Подготовьте векторы признаков — используйте пять ключевых показателей из датасета `df_customers.csv`: `recency`, `total_orders`, `total_sales`, `return_rate` и `avg_discount`.
2. Выполните предобработку данных и обеспечьте защиту от утечек.
3. Используйте метод локтя, чтобы найти математически обоснованную границу количества групп.
4. Проанализируйте полученные кластеры: дайте статистическую характеристику и опишите самые яркие группы.
5. Визуализируйте положение кластеров с помощью t-SNE.



<a id="enhanced_model"></a>

##  Обучение модели на новых признаках

1. Добавьте результаты кластеризаций в данные для обучения моделей.
2. Обучите финальные версии логистической регрессии и `CatBoostClassifier` на расширенном наборе данных.
3. Рассчитайте итоговые значения метрики ROC−AUC на валидационной выборке.
   

<a id="extra"></a>

## Дополнительное задание — подбор лучшего количества кластеров

Рассмотрите количество кластеров как гиперпараметр всей системы и найти ту «степень детализации», которая даст максимальный прирост ROC-AUC.

Вы можете взять значения из списка или выбрать свои:

```python
geo_k_range = [4, 8, 12, 20]
rfm_k_range = [6, 12, 20, 30]
```



<a id="test"></a>

## Тестирование лучшей модели

На этом этапе вы должны убедиться, что выбранная модель сохраняет высокое качество на данных, которые она никогда не видела, и понять, какие факторы стали решающими для прогноза.

1. Выполните предсказание на тестовой выборке для лучшей модели. Рассчитайте итоговый ROC-AUC и сравните его с целевым показателем 0.75.
2. Постройте матрицу ошибок. Определите, какой тип ошибок совершает модель чаще: пропускает ли она реальные задержки или слишком часто выдаёт ложную тревогу?
3. Визуализируйте важность признаков вашей лучшей модели.
4. Проанализируйте позиции созданных вами признаков `geo_cluster` и `beh_cluster` в общем рейтинге. Стали ли они ключевыми факторами для предсказаний модели для модели или базовые параметры заказа (цена, время) остались приоритетными?

<a id="conclusion"></a>

##  Выводы и рекомендации

Что нужно зафиксировать:

* **Результаты моделирования.** Укажите итоговое значение ROC-AUC на тестовой выборке. Удалось ли вам достичь целевого показателя? Насколько эффективно итоговая модель справляется с выявлением задержек по сравнению с базовыми?
* **Эффективность сегментации.** Сформулируйте вывод о полезности кластеризации. Подтвердилась ли гипотеза о том, что профиль клиента и его географическое положение влияют на риск задержки? Какие именно кластеры — географические или поведенческие — оказались более информативными для модели? Добавьте в раздел визуализации, которые вы получили, работая над проектом.
* **Технический инсайт.** Если вы экспериментировали с разным количеством  кластеров как с гиперпараметром, то укажите, какое количество кластеров K оказалось оптимальным. Кратко поясните, почему слишком большое K может вредить качеству прогноза.
* **Бизнес-рекомендации.** Предложите, как CargaPronto может использовать вашу модель в реальных операциях.